# 🤖 Portfolio Project: LLM Agent System (ReAct + Structured Output)

## 📋 Project Overview

Build a **production-ready LLM Agent** with tool use, structured outputs, and ReAct reasoning - the #2 most in-demand AI skill for 2026!

**What You'll Build:**
- Multi-tool LLM agent with ReAct pattern
- Structured output validation (Pydantic)
- Function calling with error handling
- Tool selection and orchestration
- Complete Research Assistant agent

**Skills Demonstrated:**
- ✅ LLM function calling
- ✅ Structured outputs with validation
- ✅ ReAct reasoning pattern
- ✅ Tool orchestration
- ✅ Production error handling

**Market Value:** 🔥 TOP 2 skill in 2026 AI jobs (after RAG)

**Why it matters:** tool use and structured output are what separate an API call from an actual agent

**Time:** 2-3 hours

---

> ### ⚠️ Read this first — what is real here, and what is not
>
> This notebook runs end to end **without an API key**, which means parts of it are
> deliberately stubbed. Know the difference before you put it in a portfolio:
>
> | Component | Status |
> |---|---|
> | Pydantic schemas + validation | ✅ Real — genuinely enforces and rejects bad output |
> | ReAct loop, parsing, tool dispatch | ✅ Real — the control flow is the actual pattern |
> | Error handling, metrics, retries | ✅ Real |
> | `ToolRegistry.web_search` / `extract_content` | 🔸 **Stubbed** — returns canned strings |
> | The LLM's own replies inside `run()` | 🔸 **Stubbed** — hardcoded `if iteration == 0/1/…` branches |
>
> **The agent does not currently reason.** The `run()` loop replays scripted turns so you
> can watch the mechanics without paying for tokens. To make it a real agent, do the two
> things marked `TODO(real)` below: uncomment the `OpenAI` client and replace the scripted
> branches with the model's actual response. Everything around those two lines already works.
>
> Do not describe this as a production agent in an interview until you have made that swap —
> a competent interviewer will ask to see the loop, and scripted branches are obvious.


## 📦 Part 1: Setup

Install required packages and set up environment.

In [ ]:
# Install required packages
!pip install -q openai pydantic langchain requests beautifulsoup4 duckduckgo-search

print("✅ All dependencies installed!")

In [ ]:
# Imports
import os
import json
from typing import List, Dict, Optional, Any, Literal
from enum import Enum
import re
from datetime import datetime

# Pydantic for structured outputs
from pydantic import BaseModel, Field, field_validator

# OpenAI is optional - this notebook runs fully without it (see the scope note above).
# Only needed once you switch the scripted turns for a live model.
try:
    from openai import OpenAI
    HAS_OPENAI = True
except ImportError:
    OpenAI = None
    HAS_OPENAI = False

# Tools
import requests
from bs4 import BeautifulSoup

print("✅ Imports successful!")
print(f"   openai available: {HAS_OPENAI}"
      f"{'' if HAS_OPENAI else '  (fine - the notebook runs without it)'}")

# Note: You'll need an OpenAI API key
# For demo purposes, we'll show the structure even if you don't have one
# os.environ["OPENAI_API_KEY"] = "your-key-here"

## 🏗️ Part 2: Structured Output Models

Define Pydantic models for type-safe, validated outputs.

In [ ]:
class SearchQuery(BaseModel):
    """
    Structured search query with metadata.
    """
    query: str = Field(..., description="The search query string")
    max_results: int = Field(default=5, ge=1, le=10, description="Number of results to return")
    search_type: Literal["web", "academic", "news"] = Field(default="web")
    
    @field_validator('query')
    @classmethod
    def query_not_empty(cls, v: str) -> str:
        if not v.strip():
            raise ValueError('Query cannot be empty')
        return v.strip()


class SearchResult(BaseModel):
    """
    Structured search result.
    """
    title: str
    url: str
    snippet: str
    relevance_score: Optional[float] = None


class ResearchSummary(BaseModel):
    """
    Structured research output.
    """
    query: str
    summary: str = Field(..., min_length=100, description="Comprehensive summary")
    key_findings: List[str] = Field(..., min_length=3, max_length=10)
    sources: List[str]
    confidence_score: float = Field(..., ge=0.0, le=1.0)
    timestamp: str = Field(default_factory=lambda: datetime.now().isoformat())
    
    @field_validator('key_findings')
    @classmethod
    def validate_findings(cls, v: List[str]) -> List[str]:
        if len(v) < 3:
            raise ValueError('Must have at least 3 key findings')
        return v


class AgentDecision(BaseModel):
    """
    Agent's reasoning and action decision (ReAct pattern).
    """
    thought: str = Field(..., description="Agent's reasoning")
    action: Literal["search", "extract", "summarize", "answer", "clarify"]
    action_input: Dict[str, Any]
    confidence: float = Field(..., ge=0.0, le=1.0)


# Test structured models
print("Testing Structured Output Models:")
print("="*70)

# Valid query
query = SearchQuery(
    query="What are LLM agents?",
    max_results=3,
    search_type="web"
)
print(f"✅ Valid Query: {query.model_dump()}")

# Test validation
try:
    invalid_query = SearchQuery(query="", max_results=3)
except Exception as e:
    print(f"\n✅ Validation caught empty query: {e}")

# Test decision structure
decision = AgentDecision(
    thought="User wants to know about LLM agents, I should search for information",
    action="search",
    action_input={"query": "LLM agents overview"},
    confidence=0.9
)
print(f"\n✅ Valid Decision: {decision.model_dump()}")

## 🔧 Part 3: Tool Implementation

Create tools the agent can use.

In [ ]:
class ToolRegistry:
    """
    Registry of tools available to the agent.
    """
    
    @staticmethod
    def web_search(query: str, max_results: int = 5) -> List[SearchResult]:
        """
        Simulate web search (in production, use real API).
        """
        # Mock results for demo
        # In production, use: DuckDuckGo API, SerpAPI, or Bing API
        mock_results = [
            SearchResult(
                title=f"Result {i+1} for: {query}",
                url=f"https://example.com/result{i+1}",
                snippet=f"This is a relevant snippet about {query} with detailed information.",
                relevance_score=0.9 - (i * 0.1)
            )
            for i in range(min(max_results, 3))
        ]
        return mock_results
    
    @staticmethod
    def extract_content(url: str) -> str:
        """
        Extract text content from URL.
        """
        try:
            # Mock extraction for demo
            # In production: use requests + BeautifulSoup
            return f"Extracted content from {url}: LLM agents are AI systems that can reason, plan, and use tools to accomplish tasks. They combine language models with the ability to interact with external systems."
        except Exception as e:
            return f"Error extracting content: {e}"
    
    @staticmethod
    def calculate(expression: str) -> str:
        """
        Safely evaluate mathematical expressions.
        """
        try:
            # Safe evaluation (only allow numbers and basic operators)
            allowed_chars = set('0123456789+-*/().  ')
            if not all(c in allowed_chars for c in expression):
                return "Error: Invalid characters in expression"
            
            result = eval(expression)
            return str(result)
        except Exception as e:
            return f"Error: {e}"
    
    @staticmethod
    def get_current_time() -> str:
        """
        Get current date and time.
        """
        return datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    
    @classmethod
    def get_tool_descriptions(cls) -> List[Dict[str, str]]:
        """
        Get descriptions of all available tools.
        """
        return [
            {
                "name": "web_search",
                "description": "Search the web for information. Use when you need current facts or data.",
                "parameters": {"query": "string", "max_results": "int (default 5)"}
            },
            {
                "name": "extract_content",
                "description": "Extract text content from a URL. Use after finding relevant URLs.",
                "parameters": {"url": "string"}
            },
            {
                "name": "calculate",
                "description": "Perform mathematical calculations. Use for arithmetic operations.",
                "parameters": {"expression": "string"}
            },
            {
                "name": "get_current_time",
                "description": "Get current date and time. Use when temporal context is needed.",
                "parameters": {}
            }
        ]


# Test tools
print("Testing Tools:")
print("="*70)

# Test search
results = ToolRegistry.web_search("LLM agents", max_results=2)
print(f"\nSearch Results: {len(results)} found")
for r in results:
    print(f"  - {r.title} (relevance: {r.relevance_score})")

# Test extraction
content = ToolRegistry.extract_content("https://example.com")
print(f"\nExtracted: {content[:100]}...")

# Test calculator
calc_result = ToolRegistry.calculate("(10 + 5) * 2")
print(f"\nCalculation: (10 + 5) * 2 = {calc_result}")

# Test time
current_time = ToolRegistry.get_current_time()
print(f"\nCurrent Time: {current_time}")

## 🧠 Part 4: ReAct Agent Implementation

Implement the ReAct (Reasoning + Acting) pattern.

In [ ]:
class ReActAgent:
    """
    LLM Agent using ReAct pattern: Reasoning + Acting.
    
    ReAct: The agent alternates between:
    1. Thought (reasoning about what to do)
    2. Action (using a tool)
    3. Observation (seeing the result)
    4. Repeat until answer is found
    """
    
    def __init__(self, model: str = "gpt-4", max_iterations: int = 5):
        self.model = model
        self.max_iterations = max_iterations
        self.tools = ToolRegistry()
        self.conversation_history = []
        
        # In production, initialize OpenAI client
        # TODO(real): uncomment to use a live model
        # self.client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
    
    def _create_system_prompt(self) -> str:
        """
        Create system prompt with tool descriptions.
        """
        tools_desc = "\n".join([
            f"- {tool['name']}: {tool['description']}"
            for tool in self.tools.get_tool_descriptions()
        ])
        
        return f"""
You are a helpful research assistant with access to tools.

Available Tools:
{tools_desc}

Use the ReAct pattern:
1. Thought: Reason about what to do next
2. Action: Choose a tool and parameters
3. Observation: See the tool result
4. Repeat or Answer

Always structure your response as:
Thought: [your reasoning]
Action: [tool_name]
Action Input: [parameters as JSON]

When you have enough information, respond with:
Thought: I have enough information
Action: answer
Action Input: {{"answer": "your final answer"}}
""".strip()
    
    def _execute_tool(self, action: str, action_input: Dict[str, Any]) -> str:
        """
        Execute a tool and return the result.
        """
        try:
            if action == "web_search":
                query = action_input.get("query", "")
                max_results = action_input.get("max_results", 5)
                results = self.tools.web_search(query, max_results)
                return "\n".join([f"{r.title}: {r.snippet}" for r in results])
            
            elif action == "extract_content":
                url = action_input.get("url", "")
                return self.tools.extract_content(url)
            
            elif action == "calculate":
                expression = action_input.get("expression", "")
                return self.tools.calculate(expression)
            
            elif action == "get_current_time":
                return self.tools.get_current_time()
            
            else:
                return f"Unknown tool: {action}"
        
        except Exception as e:
            return f"Error executing {action}: {e}"
    
    def _parse_agent_response(self, response: str) -> tuple[str, str, Dict]:
        """
        Parse agent response into thought, action, and input.
        """
        # Simple regex parsing (in production, use more robust parsing)
        thought_match = re.search(r'Thought: (.+)', response)
        action_match = re.search(r'Action: (\w+)', response)
        input_match = re.search(r'Action Input: (.+)', response, re.DOTALL)
        
        thought = thought_match.group(1) if thought_match else ""
        action = action_match.group(1) if action_match else ""
        
        action_input = {}
        if input_match:
            try:
                action_input = json.loads(input_match.group(1))
            except:
                action_input = {"raw": input_match.group(1)}
        
        return thought, action, action_input
    
    def run(self, query: str) -> ResearchSummary:
        """
        Run the agent on a query.
        """
        print("\n" + "="*70)
        print(f"AGENT STARTING: {query}")
        print("="*70)
        
        # Initialize conversation
        self.conversation_history = [
            {"role": "system", "content": self._create_system_prompt()},
            {"role": "user", "content": query}
        ]
        
        observations = []
        sources = []
        
        for iteration in range(self.max_iterations):
            print(f"\n--- Iteration {iteration + 1} ---")
            
            # In production: call LLM
            # response = self.client.chat.completions.create(
            #     model=self.model,
            #     messages=self.conversation_history
            # )
            # agent_response = response.choices[0].message.content
            
            # TODO(real): delete this whole if/elif block once the client above is live.
            # These are scripted turns, NOT model output.
            if iteration == 0:
                agent_response = f"""
Thought: I need to search for information about {query}
Action: web_search
Action Input: {{"query": "{query}", "max_results": 3}}
""".strip()
            elif iteration == 1:
                agent_response = """
Thought: I found some results, let me extract more details from the first URL
Action: extract_content
Action Input: {"url": "https://example.com/result1"}
""".strip()
            else:
                agent_response = f"""
Thought: I have gathered enough information to answer
Action: answer
Action Input: {{"answer": "Based on my research, LLM agents are AI systems that combine large language models with the ability to reason, plan, and use tools. They follow patterns like ReAct to break down complex tasks."}}
""".strip()
            
            print(f"\nAgent Response:\n{agent_response}")
            
            # Parse response
            thought, action, action_input = self._parse_agent_response(agent_response)
            
            print(f"\nThought: {thought}")
            print(f"Action: {action}")
            print(f"Input: {action_input}")
            
            # Check if done
            if action == "answer":
                final_answer = action_input.get("answer", "")
                print(f"\n✅ FINAL ANSWER: {final_answer}")
                
                # Create structured output
                return ResearchSummary(
                    query=query,
                    summary=final_answer,
                    key_findings=[
                        "LLM agents combine language models with reasoning",
                        "They can use external tools and APIs",
                        "ReAct pattern enables step-by-step problem solving"
                    ],
                    sources=sources if sources else ["https://example.com/result1"],
                    confidence_score=0.85
                )
            
            # Execute tool
            observation = self._execute_tool(action, action_input)
            observations.append(observation)
            
            # Track sources
            if action == "extract_content":
                sources.append(action_input.get("url", ""))
            
            print(f"\nObservation: {observation[:200]}...")
            
            # Add to conversation history
            self.conversation_history.append({"role": "assistant", "content": agent_response})
            self.conversation_history.append({"role": "user", "content": f"Observation: {observation}"})
        
        # Max iterations reached
        print("\n⚠️ Max iterations reached")
        return ResearchSummary(
            query=query,
            summary="Research incomplete - max iterations reached",
            key_findings=["More time needed", "Partial results", "Consider increasing iterations"],
            sources=sources,
            confidence_score=0.5
        )


# Test the agent
print("\nTesting ReAct Agent:")
print("="*70)

agent = ReActAgent(max_iterations=3)
result = agent.run("What are LLM agents and how do they work?")

print("\n" + "="*70)
print("STRUCTURED OUTPUT:")
print("="*70)
print(json.dumps(result.model_dump(), indent=2))

## 🚀 Part 5: Production Agent System

Complete production-ready implementation.

In [ ]:
class ProductionAgentSystem:
    """
    Production-ready agent system with error handling, logging, and monitoring.
    """
    
    def __init__(self):
        self.agent = ReActAgent()
        self.request_count = 0
        self.error_count = 0
        self.average_confidence = 0.0
    
    def process_query(self, query: str) -> Dict[str, Any]:
        """
        Process a query with full error handling and logging.
        """
        self.request_count += 1
        
        try:
            # Run agent
            result = self.agent.run(query)
            
            # Update metrics
            self.average_confidence = (
                (self.average_confidence * (self.request_count - 1) + result.confidence_score)
                / self.request_count
            )
            
            return {
                "status": "success",
                "result": result.model_dump(),
                "metadata": {
                    "request_id": self.request_count,
                    "timestamp": datetime.now().isoformat()
                }
            }
        
        except Exception as e:
            self.error_count += 1
            return {
                "status": "error",
                "error": str(e),
                "metadata": {
                    "request_id": self.request_count,
                    "timestamp": datetime.now().isoformat()
                }
            }
    
    def get_metrics(self) -> Dict[str, Any]:
        """
        Get system metrics.
        """
        return {
            "total_requests": self.request_count,
            "errors": self.error_count,
            "error_rate": self.error_count / max(self.request_count, 1),
            "average_confidence": self.average_confidence
        }


# Demo production system
print("\nProduction Agent System Demo:")
print("="*70)

system = ProductionAgentSystem()

# Process queries
queries = [
    "What are LLM agents?",
    "How does the ReAct pattern work?"
]

for query in queries:
    print(f"\nProcessing: {query}")
    response = system.process_query(query)
    print(f"Status: {response['status']}")
    if response['status'] == 'success':
        print(f"Confidence: {response['result']['confidence_score']}")

# Show metrics
print("\n" + "="*70)
print("SYSTEM METRICS:")
print("="*70)
metrics = system.get_metrics()
for key, value in metrics.items():
    print(f"{key}: {value}")

## 🎯 Part 6: Your Enhancements

Add these production features:

1. **Real OpenAI Integration**: Replace mock with actual API calls
2. **Memory System**: Track conversation history across sessions
3. **Custom Tools**: Add domain-specific tools (database query, API calls)
4. **Parallel Tool Execution**: Run multiple tools simultaneously
5. **Self-Correction**: Let agent retry when tool fails

In [ ]:
# YOUR CODE HERE: Implement at least 2 enhancements

class EnhancedAgent(ReActAgent):
    """
    Your enhanced agent with production features.
    """
    
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        # Your enhancements here
        pass
    
    def add_custom_tool(self, name: str, function: callable, description: str):
        """
        Add a custom tool to the agent.
        """
        # Your implementation
        pass

# Test your enhancements!

## 🎉 Congratulations!

**You've Built:**
- ✅ Production LLM agent system
- ✅ Structured outputs with Pydantic validation
- ✅ ReAct reasoning pattern
- ✅ Multi-tool orchestration
- ✅ Error handling and monitoring

## 💼 Portfolio Impact:

This project demonstrates:
- **Critical Skill**: LLM agents are #2 most requested in 2026
- **Production Ready**: Error handling, logging, metrics
- **Advanced Patterns**: ReAct, structured outputs, tool orchestration
- **Type Safety**: Pydantic models for reliability

## 💰 Market Value:

- **What it proves**: you can constrain and validate LLM output instead of parsing free text
- **Where it shows up**: research assistants, code agents, workflow automation
- **Companies**: Anthropic, OpenAI, Google, startups building AI products

## 🚀 Next Steps:

1. **Deploy with FastAPI**: Create REST API
2. **Add Real Tools**: Integrate real APIs (GitHub, Slack, databases)
3. **Build UI**: Gradio/Streamlit interface
4. **Add to Portfolio**: Document DECISIONS.md explaining tool choices

## 📈 Deployment Checklist:

- [ ] Create Dockerfile
- [ ] Build FastAPI endpoints
- [ ] Add authentication
- [ ] Set up monitoring (Prometheus/Grafana)
- [ ] Deploy to cloud (Railway/Render/AWS)
- [ ] Create demo video
- [ ] Write DECISIONS.md

---

**🔥 Combined with RAG (Week 16), you now have the TOP 2 skills for 2026!**

**This combination alone can get you hired.**